# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ishkaklotar03/ML-Internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

**Task type: Classification (binary) that I use as a ranking.**

My lane is Lane 2, Refresh / Content Opportunity Scoring. For each page I predict one yes/no question: is this page's trend going "down"? The model gives each page a probability, and I sort pages by that probability to build a review queue.

Why classification, and not the other three:
- **Not clustering:** clustering has no label. I do have a label (`trend_direction == "down"`), so supervised learning fits better.
- **Not pure scoring/ranking:** I have no "true" ordered list of pages to learn from. I learn a yes/no label first, then use its probability to rank.
- **Classification:** two classes (down / not down), and the probability turns into a ranked queue for reviewers.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print("Task type: binary classification, used to rank pages")
print("Class 1: trend_direction == 'down'")
print("Class 0: trend_direction is not 'down'")

Task type: binary classification, used to rank pages
Class 1: trend_direction == 'down'
Class 0: trend_direction is not 'down'


**Target (proxy label):** `is_declining_label = 1` when `trend_direction == "down"`, otherwise 0.

**Where the label comes from:** it is a *defined rule on an observed column*, not a future outcome. `trend_direction` is a bucket calculated from the current window. So this is a **beginner proxy label**: a stand-in for the thing I really care about (a lasting decline). It could also reflect seasonality, consolidation into another page, or noise, so I will describe it as directional.

**Stronger target for later (Week 4+):** a future-window label on the warehouse daily table, for example "features from the prior 90 days -> decline over the next 30 days".

**Features (inputs):** only observable signals known at the decision point, such as `impressions_90d`, `sessions_90d`, `avg_position`, `ctr`, `content_age_days`, `word_count`, and `days_since_last_update`.

**Not used as features:** the target itself, and any product decision score (the data ships observable signals only, so there is nothing to strip out).

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
print("Target: is_declining_label = (trend_direction == 'down')")
print("Label type: proxy from the current window, not a future outcome")
print("Stronger label planned: prior 90 days of features -> next 30 days decline")

Target: is_declining_label = (trend_direction == 'down')
Label type: proxy from the current window, not a future outcome
Stronger label planned: prior 90 days of features -> next 30 days decline


**Main metric: Precision@50** (with average precision as a second check).

Why: reviewers can only check a limited number of pages, for example 50. Precision@50 asks: of the top 50 pages my ranking says to review first, how many actually have the "down" label? That matches how the list is really used. Plain accuracy would hide the ranking quality.

**What "good" means:** precision@50 clearly higher than a simple rule baseline, measured on clients the model never saw (client-holdout validation).

**Reference from the lane guide (not computed by me):** on this slice, baseline rules got 0.240 and a random forest got 0.740. I will not assume my own result matches. I will re-earn it.

**Action it supports:** which pages a content team should review first for refresh, a title/meta rewrite, or monitoring. A high score means "review this first", not "this will recover if edited".

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Precision@K helper: of the top K ranked pages, how many are truly positive?
def precision_at_k(y_true, scores, k=50):
    order = scores.sort_values(ascending=False).index[:k]
    return y_true.loc[order].mean()

print("Metric: precision@50 (review capacity = 50 pages)")
print("Defined helper: precision_at_k(y_true, scores, k=50)")

Metric: precision@50 (review capacity = 50 pages)
Defined helper: precision_at_k(y_true, scores, k=50)


**Unit of analysis: one row = one content page (`content_id`) within a client, measured over a 90-day window.**

Each row carries observable signals (impressions, sessions, position, CTR, age, word count) and the `trend_direction` bucket. I add one column, `is_declining_label`, which is 1 for "down" and 0 otherwise. Next to each page it would look like a column of 0s and 1s.

I keep only rows with `impressions_90d > 0` and `content_age_days >= 90`, and one row per `content_id`, as the starter pipeline does.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import os
import pandas as pd
from pathlib import Path

# Clone the repo in Colab if the data is not already there
if not Path("ML-Internship").exists() and not Path("data/raw/content_refresh_anonymized.csv").exists():
    !git clone https://github.com/ishkaklotar03/ML-Internship.git

candidates = [
    Path("ML-Internship/data/raw/content_refresh_anonymized.csv"),
    Path("../../data/raw/content_refresh_anonymized.csv"),
    Path("data/raw/content_refresh_anonymized.csv"),
]
path = next((p for p in candidates if p.exists()), None)
if path is None:
    raise FileNotFoundError("content_refresh_anonymized.csv not found")

df = pd.read_csv(path)

# Same eligibility rule as the starter pipeline
df = df[(df["impressions_90d"] > 0) & (df["content_age_days"] >= 90)]
df = df.drop_duplicates(subset="content_id").reset_index(drop=True)

# Proxy target
df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)

print("Rows:", len(df), "| Clients:", df["client_id"].nunique())
print("Share labelled down:", round(df["is_declining_label"].mean(), 3))
cols = ["content_id", "client_id", "impressions_90d", "sessions_90d",
        "avg_position", "ctr", "content_age_days", "trend_direction",
        "is_declining_label"]
df[cols].head(10)

Cloning into 'ML-Internship'...
remote: Enumerating objects: 136, done.
remote: Counting objects: 100% (136/136), done.
remote: Compressing objects: 100% (89/89), done.
remote: Total 136 (delta 47), reused 99 (delta 31), pack-reused 0 (from 0)
Receiving objects: 100% (136/136), 1.87 MiB | 11.68 MiB/s, done.
Resolving deltas: 100% (47/47), done.
Rows: 30000 | Clients: 32
Share labelled down: 0.542


,content_id,client_id,impressions_90d,sessions_90d,avg_position,ctr,content_age_days,trend_direction,is_declining_label
0,content_304f48230142,client_f369cb89fc,3803,17,10.6,0.76,187,down,1
1,content_a1fb4e703a9e,client_4e07408562,15320,9,20.3,0.05,445,down,1
2,content_9aa793d4d895,client_7f2253d7e2,12581,11,36.5,0.09,141,down,1
3,content_331d6c4de07b,client_19581e27de,11751,78,6.2,0.49,463,stable,0
4,content_d99b7a2d90ca,client_3fdba35f04,19140,145,44.0,0.13,263,down,1
5,content_d4084a4bc775,client_f369cb89fc,3970,5,8.5,0.03,147,down,1
6,content_9a34b442b552,client_8722616204,20,1,7.0,0.00,90,down,1
7,content_a63219c6e95a,client_19581e27de,1724,28,21.2,0.06,445,stable,0
8,content_5e6c160719bc,client_6208ef0f77,32574,68,46.0,0.09,90,down,1
9,content_c27558df2b0c,client_19581e27de,1240,3,4.9,0.16,257,down,1


A single fixed rule is too simple here. For example, "if the page is old and has many impressions, flag it" treats every signal the same for every page. In practice:

- Decline depends on several signals together (visibility, position, CTR, age, engagement), not on one.
- The same signal means different things in different cases. A low CTR at position 2 is a different problem from a low CTR at position 15.
- Visibility is very concentrated (a small share of pages holds most impressions), so one global threshold fits few pages well.

The starter baseline rules reach a lower precision@50 than the learned models on this slice, according to the lane guide. I will check this myself in later weeks. For now I only claim that a simple rule is a weak reference point. ML is a candidate to rank better, as decision-support, not a guarantee.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# One simple rule: old page with many impressions (stale_visible_page style)
if "days_since_last_update" in df.columns:
    rule = (df["days_since_last_update"] >= 180) & (df["impressions_90d"] >= 500)
else:
    rule = (df["content_age_days"] >= 180) & (df["impressions_90d"] >= 500)

df["rule_flag"] = rule.astype(int)

print("Pages flagged by the rule:", int(df["rule_flag"].sum()))
print(pd.crosstab(df["rule_flag"], df["is_declining_label"], margins=True))
print("Share 'down' among flagged:  ", round(df.loc[df["rule_flag"] == 1, "is_declining_label"].mean(), 3))
print("Share 'down' among unflagged:", round(df.loc[df["rule_flag"] == 0, "is_declining_label"].mean(), 3))

Pages flagged by the rule: 17
is_declining_label      0      1    All
rule_flag                              
0                   13737  16246  29983
1                       1     16     17
All                 13738  16262  30000
Share 'down' among flagged:   0.941
Share 'down' among unflagged: 0.542


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.